In [1]:
# import dependencies
import pandas as pd

# Merge all data

The last step before analysis

In [2]:
# open relevant csv's
gis_df = pd.read_csv("../data/cleaned/town_flood_risk.csv")
acs_df = pd.read_csv("../data/cleaned/acs_summary.csv")
nfip_df = pd.read_csv("../data/cleaned/nfip_summary.csv")
hma_df = pd.read_csv("../data/cleaned/fema_hma_town_level.csv")

In [3]:
print(gis_df.shape)
gis_df.columns

(256, 7)


Index(['GEOID', 'town_name', 'aland_sqm', 'awater_sqm', 'town_area_sqm',
       'pct_river_corridor', 'pct_high_risk_NFHL'],
      dtype='object')

In [4]:
print(acs_df.shape)
acs_df.columns

(256, 24)


Index(['GEOID', 'town_name', 'median_income', 'median_income_moe',
       'total_population', 'total_population_moe', 'percent_elderly',
       'percent_elderly_moe', 'median_year_house_built',
       'median_year_house_built_moe', 'pct_no_vehicle', 'pct_no_vehicle_moe',
       'occupied_housing_units', 'occupied_housing_units_moe',
       'pct_renter_occupied', 'pct_renter_occupied_moe',
       'pct_bachelors_or_higher', 'pct_bachelors_or_higher_moe',
       'pct_below_poverty', 'pct_below_poverty_moe', 'percent_with_disability',
       'percent_with_disability_moe', 'pct_mobile_home',
       'pct_mobile_home_moe'],
      dtype='object')

In [5]:
print(nfip_df.shape)
nfip_df.columns

(256, 18)


Index(['GEOID', 'town_name', 'total_population', 'occupied_housing_units',
       'nfip_claims', 'total_nfip_claims_paid', '2011_2022_nfip_claims',
       '2023_plus_nfip_claims', 'pre_2011_nfip_claims',
       '2011_2022_nfip_claims_count', '2023_plus_nfip_claims_count',
       'pre_2011_nfip_claims_count', 'claims_paid_per_capita',
       'claims_paid_per_housing_unit', 'irene_policies', 'flood2023_policies',
       'today_policies', 'current_insurance_penetration'],
      dtype='object')

In [6]:
print(hma_df.shape)
hma_df.columns

(256, 37)


Index(['GEOID', 'town_name', 'total_population', 'occupied_housing_units',
       'funding_acquisition_buyouts', 'funding_admin',
       'funding_ecosystem_restoration', 'funding_equipment',
       'funding_flood_control', 'funding_infrastructure_protection',
       'funding_infrastructure_stormwater', 'funding_other',
       'funding_planning', 'funding_structure_protection',
       'federalShareObligated', 'numberOfFinalProperties',
       'numberOfProperties', '2011_2022', '2023_plus', 'pre_2011',
       'cost_per_property', 'cost_per_final_property',
       'share_acquisition_buyouts', 'share_admin',
       'share_ecosystem_restoration', 'share_equipment', 'share_flood_control',
       'share_infrastructure_protection', 'share_infrastructure_stormwater',
       'share_other', 'share_planning', 'share_structure_protection',
       'funding_per_capita', 'log_funding_per_capita',
       'funding_per_occupied_unit', 'log_funding_per_occupied_unit',
       'has_funding'],
      dtype='o

In [7]:
# safety check before merging GIS and ACS df's

# check for missing GEOIDs/town_names between ACS and GIS
acs_keys = set(zip(acs_df["GEOID"], acs_df["town_name"]))
gis_keys = set(zip(gis_df["GEOID"], gis_df["town_name"]))

acs_not_in_gis = acs_keys - gis_keys
gis_not_in_acs = gis_keys - acs_keys

if acs_not_in_gis:
    print("Towns in ACS but not in GIS:", acs_not_in_gis)
if gis_not_in_acs:
    print("Towns in GIS but not in ACS:", gis_not_in_acs)
if not acs_not_in_gis and not gis_not_in_acs:
    print("All GEOID/town_name pairs match between ACS and GIS.")

# check for duplicates
if acs_df.duplicated(subset=["GEOID", "town_name"]).any():
    print("Warning: Duplicate GEOID/town_name pairs in ACS.")
if gis_df.duplicated(subset=["GEOID", "town_name"]).any():
    print("Warning: Duplicate GEOID/town_name pairs in GIS.")

All GEOID/town_name pairs match between ACS and GIS.


In [8]:
# safety check before merging ACS and NFIP df's
# get keys for checking
acs_keys = acs_df[["GEOID", "town_name", "total_population", "occupied_housing_units"]]
nfip_keys = nfip_df[
    ["GEOID", "town_name", "total_population", "occupied_housing_units"]
]

# merge to find mismatches
merged_keys = pd.merge(
    acs_keys,
    nfip_keys,
    on=["GEOID", "town_name", "total_population", "occupied_housing_units"],
    how="outer",
    indicator=True,
)

# check for mismatches
mismatches = merged_keys[merged_keys["_merge"] != "both"]

if not mismatches.empty:
    print(
        "Warning: Found mismatches in town names or population/housing units between ACS and NFIP:"
    )
    print(mismatches)
else:
    print(
        "All town names and population/housing unit values match between ACS and NFIP."
    )

All town names and population/housing unit values match between ACS and NFIP.


In [9]:
# safety check before merging ACS and HMA df's

# get keys for checking
acs_keys = acs_df[["GEOID", "town_name", "total_population", "occupied_housing_units"]]
hma_keys = hma_df[["GEOID", "town_name", "total_population", "occupied_housing_units"]]

# merge to find mismatches
merged_keys = pd.merge(
    acs_keys,
    hma_keys,
    on=["GEOID", "town_name", "total_population", "occupied_housing_units"],
    how="outer",
    indicator=True,
)

# check for mismatches
mismatches = merged_keys[merged_keys["_merge"] != "both"]

if not mismatches.empty:
    print(
        "Warning: Found mismatches in population or housing units between ACS and HMA:"
    )
    print(mismatches)
else:
    print("All population and housing unit values match between ACS and HMA.")

All population and housing unit values match between ACS and HMA.


In [10]:
# merge all dataframes

# merge GIS and ACS first
merged = pd.merge(gis_df, acs_df, on=["GEOID", "town_name"], how="outer")

# merge in NFIP data
merged = pd.merge(
    merged,
    nfip_df,
    on=["GEOID", "town_name", "total_population", "occupied_housing_units"],
    how="outer",
)

# merge in HMA data
merged = pd.merge(
    merged,
    hma_df,
    on=["GEOID", "town_name", "total_population", "occupied_housing_units"],
    how="outer",
)

In [11]:
merged.shape

(256, 76)

In [12]:
merged.columns

Index(['GEOID', 'town_name', 'aland_sqm', 'awater_sqm', 'town_area_sqm',
       'pct_river_corridor', 'pct_high_risk_NFHL', 'median_income',
       'median_income_moe', 'total_population', 'total_population_moe',
       'percent_elderly', 'percent_elderly_moe', 'median_year_house_built',
       'median_year_house_built_moe', 'pct_no_vehicle', 'pct_no_vehicle_moe',
       'occupied_housing_units', 'occupied_housing_units_moe',
       'pct_renter_occupied', 'pct_renter_occupied_moe',
       'pct_bachelors_or_higher', 'pct_bachelors_or_higher_moe',
       'pct_below_poverty', 'pct_below_poverty_moe', 'percent_with_disability',
       'percent_with_disability_moe', 'pct_mobile_home', 'pct_mobile_home_moe',
       'nfip_claims', 'total_nfip_claims_paid', '2011_2022_nfip_claims',
       '2023_plus_nfip_claims', 'pre_2011_nfip_claims',
       '2011_2022_nfip_claims_count', '2023_plus_nfip_claims_count',
       'pre_2011_nfip_claims_count', 'claims_paid_per_capita',
       'claims_paid_per_hou

In [13]:
# export merged dataframe to csv
merged.to_csv("../data/cleaned/town_level_merged_for_eda.csv", index=False)